# 1 · Preprocesamiento de datos

> **Tipo de ML:** `supervisado`

## 1. Imports

In [4]:
import pandas as pd
import numpy as np
import joblib

from cyberforest.data.make_dataset import load_data
from cyberforest.features.build_features import preprocess_data
from cyberforest.utils.paths import PROCESSED_DATA_DIR, ARTIFACTS_DIR


## 2. Cargar datos crudos

In [5]:
df = load_data()
print(df.shape)
df.head()


--> Cargando Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv...
--> Cargando Monday-WorkingHours.pcap_ISCX.csv...
--> Cargando Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv...
--> Cargando Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv...
--> Cargando Tuesday-WorkingHours.pcap_ISCX.csv...
--> Cargando Wednesday-workingHours.pcap_ISCX.csv...

    Total combinado: (2413965, 79)
    Clases: {'BENIGN': 1986312, 'DoS Hulk': 231073, 'PortScan': 158930, 'DoS GoldenEye': 10293, 'FTP-Patator': 7938, 'SSH-Patator': 5897, 'DoS slowloris': 5796, 'DoS Slowhttptest': 5499, 'Web Attack � Brute Force': 1507, 'Web Attack � XSS': 652, 'Infiltration': 36, 'Web Attack � Sql Injection': 21, 'Heartbleed': 11}
(2413965, 79)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,22,1266342,41,44,2664,6954,456,0,64.975610,109.864573,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,22,1319353,41,44,2664,6954,456,0,64.975610,109.864573,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,22,160,1,1,0,0,0,0,0.000000,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,22,1303488,41,42,2728,6634,456,0,66.536585,110.129945,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,35396,77,1,2,0,0,0,0,0.000000,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


## 3. Preprocesar


Indica la columna objetivo (`TARGET_COL`) y el tipo de scaler.


In [6]:

TARGET_COL = 'Label'
SCALER_TYPE = 'standard'            # 'standard' | 'minmax'
X_train, X_test, y_train, y_test = preprocess_data(df, target_col=TARGET_COL, scaler_type=SCALER_TYPE)

print('X_train:', X_train.shape)
print('X_test: ', X_test.shape)
print('Balance clases (train):', y_train.value_counts(normalize=True).to_dict())



--> Preprocesando datos (target='Label', scaler='standard', PCA=None)...
    Duplicados eliminados: 281860


2026-05-09 21:22:33.335 | WARNING  | cyberforest.features.build_features:_apply_logcols:288 - logcols | 'Flow Duration' tiene valores negativos → offset 14.0000 aplicado antes de log1p
2026-05-09 21:22:33.615 | WARNING  | cyberforest.features.build_features:_apply_logcols:288 - logcols | 'Flow IAT Mean' tiene valores negativos → offset 14.0000 aplicado antes de log1p
2026-05-09 21:22:33.616 | INFO     | cyberforest.features.build_features:_apply_logcols:294 - logcols | log1p aplicado → ['Flow Duration', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Idle Mean', 'Idle Max', 'Idle Min', 'Flow IAT Mean']


    Columnas eliminadas: ['Flow Bytes/s', 'Flow Packets/s', 'Fwd Header Length.1', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Bwd PSH Flags', 'Bwd URG Flags', 'Bwd IAT Total', 'Flow IAT Max']
    Target codificado → target_encoder.joblib
    Encoders guardados → encoders.joblib  ([])
    feature_names.joblib guardado (68 features)
Scaler guardado → scaler.joblib
    Train: (1705684, 68) | Test: (426421, 68)
    Proporción clases (train): {0: 0.8612128624059322, 2: 0.09087673918498386, 3: 0.04259581493406751, 1: 0.004292706034646512, 4: 0.001021877440369963}
X_train: (1705684, 68)
X_test:  (426421, 68)
Balance clases (train): {0: 0.8612128624059322, 2: 0.09087673918498386, 3: 0.04259581493406751, 1: 0.004292706034646512, 4: 0.001021877440369963}


## 4. Guardar datos procesados

In [7]:
pd.DataFrame(X_train).to_csv(PROCESSED_DATA_DIR / 'X_train.csv', index=False)
pd.DataFrame(X_test).to_csv(PROCESSED_DATA_DIR / 'X_test.csv', index=False)
pd.Series(y_train).to_csv(PROCESSED_DATA_DIR / 'y_train.csv', index=False)
pd.Series(y_test).to_csv(PROCESSED_DATA_DIR / 'y_test.csv', index=False)

print('Datos guardados en', PROCESSED_DATA_DIR)

Datos guardados en /home/cacelas/Documentos/Proyects/PROCESO/cyberforest/data/processed


In [8]:
from cyberforest.utils.paths import PROCESSED_DATA_DIR

y_train = pd.read_csv(PROCESSED_DATA_DIR / 'y_train.csv').squeeze()
print(y_train.value_counts())

Label
0    1468957
2     155007
3      72655
1       7322
4       1743
Name: count, dtype: int64


In [9]:
from cyberforest.features.build_features import _feature_engineering
df = _feature_engineering(df)
print(df['Label'].value_counts())

Label
BENIGN         1986312
DoS             252672
PortScan        158930
Brute Force      13835
Web Attack        2216
Name: count, dtype: int64
